#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

#1. Read from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

In [0]:
df.limit(10).display()

#2. Silver Transformation

##2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##2.2 Cleaning Date

In [0]:
Date_Map = {
    "sls_order_dt"
    , "sls_ship_dt"
    , "sls_due_dt"
}

for field in Date_Map:
   df= df.withColumn(
        field
        , F.when(
            (col(field) == 0) | (length(col(field)) !=8) , None
        ).otherwise(F.to_date(col(field).cast("string"),"yyyMMdd"))        
    )

In [0]:
df.limit(10).display()

In [0]:
df.filter(col("sls_price").isNull()).display()

##2.3 Sales and Price Corrections

In [0]:
df = (
    df
    .withColumn(
        "sls_price"
        , F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0)
            , F.when(col("sls_quantity") != 0, col("sls_sales") / col("sls_quantity")).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

##2.4 Renaming Columns

In [0]:
Renamed_Map = {
    "sls_ord_num": "order_number"
    , "sls_prd_key": "product_number"
    , "sls_cust_id": "customer_id"
    , "sls_order_dt": "order_date"
    , "sls_ship_dt": "ship_date"
    , "sls_due_dt": "due_date"
    , "sls_sales": "sales_amount"
    , "sls_quantity": "quantity"
    , "sls_price": "price"
}
for old_name, new_name in Renamed_Map.items():
    df = df.withColumnRenamed(old_name, new_name)



##2.5 Sanity check of dataframe

In [0]:
df.limit(10).display()

#3. Writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

##3.1 Sanity check of silver table

In [0]:
%sql
select * from workspace.silver.crm_sales limit 10